In [1]:
import pandas as pd
import numpy as np
import re
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
#These files are from CRISPOR outputs at the specified coordinates
guides =  pd.read_table("guides_pShh_mm10-chr5-28466763-28468256.tsv")
offtargets = pd.read_table("offtargets_pShh_mm10-chr5-28466763-28468256.tsv")
name = "pShhx100"
#These are the coordinates used to get the guides
chrom = "chr5"
start = 28466763
end = 28468256

In [3]:
#Remove PAM from sequence
guides["guide"] = guides["targetSeq"].str[:-3] 
#If no G in start, add G (U6 promoter)
guides["guide"] = np.where(guides["guide"].str[0] == "G", 
                        guides["guide"],
                        "G" + guides["guide"])
#Features of guide
guides["len"] = guides.apply(lambda x: len(x["guide"]), axis=1)
guides["GC"] = guides.apply(lambda x: (x["guide"].count("G")+x["guide"].count("C"))/x["len"]*100, axis=1)
guides["noU6"] = guides.apply(lambda x: re.search("TTTT", x["guide"]) != None, axis=1)
guides["noBbsI"] = guides.apply(lambda x: re.search("GAAGAC", x["guide"]) != None, axis=1)
#Remove repeats (they are excluded from the offtarget file)
guides = guides[guides["#guideId"].isin(offtargets["guideId"])]

In [4]:
#Remove all guides with offtargets within 500 kb of Shh promoter. This can be more/less stringent
shh = 28467256
offtargets = offtargets.loc[offtargets["chrom"]==chrom]
offtargets["DistanceToShh"] = np.abs((offtargets["start"]+offtargets["end"])/2 - shh)
offtargets = offtargets.loc[offtargets["DistanceToShh"] < 500000]
guides = guides[~guides["targetSeq"].isin(offtargets["guideSeq"].tolist())]

In [5]:
#Location relative to start/end coordinates searched
guides["rel_bp"] = guides.apply(lambda x: int(re.split('(\d+)',x["#guideId"])[1]), axis=1)
guides["distCenter"] = guides.apply(lambda x: np.abs(int(re.split('(\d+)',x["#guideId"])[1])-np.abs(start-end)/2), axis=1)
#Pick the top 100 based on filtering and centering
selected = guides[~(guides["noU6"]) & ~(guides["noBbsI"]) & (guides["GC"] > 25) & (guides["mitSpecScore"] > 10) & (guides["GrafEtAlStatus"] == "GrafOK")].sort_values("distCenter").head(100).reset_index(drop=True)

In [6]:
selected["Pool name"] = name
#Add ampliciation overhangs
selected["Sequence"] = "ATGAGTCCATCAGGAAGACGGCACC" + selected["guide"] + "GTTTGGGTCTTCGCTGAGCATGATT"
selected = selected.sort_values("rel_bp").reset_index(drop=True)

In [7]:
#For genome browser
selected["chrom"] = chrom
selected["start"] = start + selected["rel_bp"]
selected["end"] = selected["start"] + 20
selected[["chrom", "start", "end"]].iloc[0:100].to_csv(f"{name}.bed", index=False, sep="\t", header=None)

In [8]:
#To order as oPools (IDT)
selected[["Pool name", "Sequence"]].iloc[0:100].to_csv(f"{name}.csv", index=False)